In [8]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: ChilePropiedades (Arriendos IV Región)
# ==============================================================================

import os
import time
import re
import random
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# --- CONFIGURACIÓN DE PÁGINAS ---
PAGINA_INICIO = 1
PAGINA_FIN = 2 

os.environ["DISPLAY"] = ":99"
os.system("pkill -9 chrome && pkill -9 chromedriver")
print(f"Entorno listo. ChilePropiedades: Páginas {PAGINA_INICIO} a {PAGINA_FIN}")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--headless") # Ejecución invisible

# Opciones Anti-Bot
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")
options.add_argument("--disable-blink-features=AutomationControlled")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # Evasión extra de detección
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })

    # ==============================================================================
    # FASE 1: RECOLECCIÓN DE URLs (Catálogo con Filtro Temprano)
    # ==============================================================================
    url_base = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- Explorando Catálogo: Página {pagina_actual} ---")
        
        # Paginación Índice 0
        indice_web = pagina_actual - 1 
        url_pagina = f"{url_base}{indice_web}"
        
        driver.get(url_pagina)
        time.sleep(random.uniform(3.5, 5.0))

        # Scroll para asentar el DOM
        for _ in range(random.randint(4, 7)):
            salto = random.randint(400, 600)
            driver.execute_script(f"window.scrollBy(0, {salto});")
            time.sleep(random.uniform(0.2, 0.5))

        print(f"Página cargada correctamente")
        
        tarjetas = driver.find_elements(By.CSS_SELECTOR, "div[id^='publication_']")
        
        for tarjeta in tarjetas:
            try:
                # Filtro temprano básico (descarta lo que dice explícitamente "Arriendo Diario" en el título)
                tipo_arriendo = tarjeta.find_element(By.CSS_SELECTOR, "h3.sub-codigo-data").text.lower()
                
                if "diario" in tipo_arriendo and "mensual" not in tipo_arriendo:
                    continue 
                
                url = tarjeta.find_element(By.TAG_NAME, "a").get_attribute("href")
                
                if not url or "/ver-publicacion/" not in url: 
                    continue

                catalogo_urls.append({
                    "url": url,
                    "ubicacion": "La Serena"
                })
            except:
                continue
                
        time.sleep(random.uniform(1.2, 2.5)) 

    catalogo_urls = list({v['url']: v for v in catalogo_urls}.values())
    print(f"\n FASE 1 COMPLETADA: {len(catalogo_urls)} propiedades válidas y únicas encontradas.")

    # ==============================================================================
    # FASE 2: INSPECCIÓN ESTRUCTURAL DE PROPIEDADES
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"[{i+1}/{len(catalogo_urls)}] Extrayendo: {propiedad['url']}")

        registro = {
            "responsable": "Constanza Torres", # Ajusta este nombre
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0, 
            "dormitorios": 0,
            "banos": 0,          # Sin 'ñ' para PySpark
            "estacionamiento": 0, # Nace binarizado
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "enlace": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            time.sleep(random.uniform(2.5, 4.0))

            # --- A. TÍTULO ---
            try:
                titulo = driver.find_element(By.CSS_SELECTOR, "h1.clp-titulo").text.strip()
                registro["titulo"] = titulo
            except:
                registro["titulo"] = "Propiedad ChilePropiedades"

            # --- B. EXTRACCIÓN TABULAR ---
            try:
                labels = driver.find_elements(By.CSS_SELECTOR, ".clp-description-label")
                values = driver.find_elements(By.CSS_SELECTOR, ".clp-description-value")

                for j in range(len(labels)):
                    label_text = labels[j].text.strip().lower()
                    value_text = values[j].text.strip().lower()

                    if label_text == "valor:": 
                        v_limpio = value_text.replace("$", "").replace(".", "").replace(" ", "").strip()
                        if v_limpio.isdigit(): registro["precio"] = int(v_limpio)
                    elif "habitaciones" in label_text:
                        nums = re.findall(r'\d+', value_text)
                        if nums: registro["dormitorios"] = int(nums[0])
                    elif "baño" in label_text:
                        nums = re.findall(r'\d+', value_text)
                        if nums: registro["banos"] = int(nums[0])
                    elif "estacionamiento" in label_text:
                        nums = re.findall(r'\d+', value_text)
                        if nums and int(nums[0]) > 0:
                            registro["estacionamiento"] = 1
                    elif "superficie total" in label_text or "superficie útil" in label_text:
                        nums = re.findall(r'\d+', value_text)
                        if nums and registro["m2"] == 0: 
                            registro["m2"] = int(nums[0])
            except Exception as e:
                pass 

            # --- C. AMENIDADES ---
            try:
                texto_descripcion = driver.find_element(By.CSS_SELECTOR, ".clp-description-box").text.lower()
            except:
                texto_descripcion = driver.find_element(By.TAG_NAME, "body").text.lower()
            
            if registro["estacionamiento"] == 0 and "estacionamiento" in texto_descripcion: registro["estacionamiento"] = 1
            if "piscina" in texto_descripcion: registro["piscina"] = 1
            if "quincho" in texto_descripcion or "asador" in texto_descripcion: registro["quincho"] = 1
            if "terraza" in texto_descripcion: registro["terraza"] = 1
            if "gimnasio" in texto_descripcion: registro["gimnasio"] = 1
            if "lavander" in texto_descripcion: registro["lavanderia"] = 1

            # --- D. FILTRO INTELIGENTE NLP (Blacklist & Whitelist) ---
            banderas_rojas = ["diario", "por noche", "por día", "temporal"]
            banderas_verdes = ["año corrido", "año de corrido", "marzo a diciembre", "mensual"]

            tiene_alerta_diaria = any(palabra in texto_descripcion for palabra in banderas_rojas)
            tiene_salvavidas = any(palabra in texto_descripcion for palabra in banderas_verdes)

            if tiene_alerta_diaria and not tiene_salvavidas:
                print("  -> Descartado: Arriendo 100% diario detectado.")
                continue

            if registro["precio"] < 100000:
                print("  -> Descartado: Precio anómalo o publicación sin valor.")
                continue

            datos_finales.append(registro)

        except Exception as e:
            print(f"  -> Error al procesar propiedad: {e}")

        driver.delete_all_cookies()
        time.sleep(random.uniform(1.5, 3.0))

except Exception as e:
    print(f"Error crítico global en Selenium: {e}")
finally:
    if driver is not None:
        driver.quit()

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB
# ==============================================================================
if datos_finales:
    print("\n Conectando a MongoDB...")
    try:
        client = MongoClient("mongodb://database:27017/") 
        db = client["ArriendosPrueba_IVRegion"]
        coleccion = db["G2_PruebaChilePropiedades2"]
        
        insertados = 0
        for d in datos_finales:
            resultado = coleccion.update_one(
                {"enlace": d["enlace"]},
                {"$set": d},
                upsert=True
            )
            if resultado.upserted_id or resultado.modified_count > 0:
                insertados += 1
                
        print(f"¡Éxito total! {insertados} propiedades guardadas/actualizadas en la base de datos.")
    except Exception as e:
        print(f" Error al guardar en MongoDB: {e}")
else:
    print("\n El bot terminó sin extraer propiedades válidas.")

🧹 Entorno listo. ChilePropiedades: Páginas 1 a 2

--- 📄 Explorando Catálogo: Página 1 ---
Página cargada correctamente

--- 📄 Explorando Catálogo: Página 2 ---
Página cargada correctamente

🏠 FASE 1 COMPLETADA: 20 propiedades válidas y únicas encontradas.
[1/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/la-serena/departamento/avenida-cruz-del-molino-460/114868469
[2/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/la-serena/departamento/la-reconquista-1736/114049640
[3/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/la-serena/departamento/avenida-libertad-201/107087297
[4/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/la-serena/departamento/av-del-mar-560/109779950
[5/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/la-serena/departamento/av-del-mar-5315/116976300
[6/20] Extrayendo: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual